# Ablation Study: Impact of Model Width on Accuracy-Efficiency Trade-off

This notebook evaluates all four U-MobileViT-Net variants (nano, base, pro,
promax) on the COCO Tea Leaf dataset to quantify the parameter-accuracy
Pareto frontier and identify the most efficient variant for edge deployment.

**Variants**: nano (0.3M), base (0.6M), pro (2.0M), promax (7.0M)
**Dataset**: COCO Tea Leaf (8 classes, multi-class)
**Metric**: Mean Intersection over Union (mIoU)


In [2]:
# --- Section 1: Environment and Imports ----------------------------------------

import sys, os
from pathlib import Path

# Locate project root by searching for .git or CLAUDE.md marker
current = Path.cwd()
project_root = current
for parent in [current] + list(current.parents):
    if (parent / ".git").exists() or (parent / "CLAUDE.md").exists():
        project_root = parent
        break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.environ.setdefault("PYTHONPATH", str(project_root))
os.chdir(str(project_root))
print(f"Project root: {project_root}")

import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt

from models.u_mobilevit_net.u_models import (
    umobilevit, UMobileViT, umobilevit_nano, umobilevit_base,
    umobilevit_pro, umobilevit_promax,
)
from models.u_mobilevit_net.configs import get_variant, UMOBILEVIT_VARIANTS
from tools.data import (
    create_dataloaders, DatasetInfo, label_to_color, denormalize,
)
from tools.training import (
    SegmentationTrainer, TrainingConfig,
    compute_class_weights, compute_pos_weight,
)
from tools.visualization import (
    configure_paper_style, plot_training_curves,
    show_dataset_samples, show_predictions,
    plot_per_class_iou, plot_confusion_matrix,
    plot_error_distribution, plot_params_vs_accuracy,
    plot_convergence_comparison, generate_results_table,
    load_training_history, load_all_results,
    TOL_PALETTE, VARIANT_COLORS,
)
from tools.evaluation import (
    compute_flops, compute_parameters, format_flops,
)

configure_paper_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    # print(f"  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


Project root: /run/media/sanng/New Volume/Seminar/U-MOBILEVIT-NET
Device: cuda
  GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
# --- Section 2: Ablation Configuration ------------------------------------------

VARIANTS = ["nano", "base", "pro", "promax"]
DATASET = "coco_leaf"
IMAGE_SIZE = (320, 320)

VARIANT_BATCH_SIZES = {"nano": 32, "base": 16, "pro": 8, "promax": 4}
VARIANT_EPOCHS = {"nano": 300, "base": 300, "pro": 200, "promax": 150}

print(f"Ablation study: {len(VARIANTS)} variants on {DATASET}")
for v in VARIANTS:
    spec = get_variant(v)
    print(f"  {v.upper():<8s}: d_model={spec['d_model']:>3d}, "
          f"blocks={spec['num_transformer_blocks']}, "
          f"~{spec['params_estimate']}, "
          f"BS={VARIANT_BATCH_SIZES[v]}, ep={VARIANT_EPOCHS[v]}")


Ablation study: 4 variants on coco_leaf
  NANO    : d_model= 48, blocks=1, ~~0.3M, BS=32, ep=300
  BASE    : d_model= 64, blocks=2, ~~0.6M, BS=16, ep=300
  PRO     : d_model=128, blocks=3, ~~2.0M, BS=8, ep=200
  PROMAX  : d_model=256, blocks=3, ~~7.0M, BS=4, ep=150


In [4]:
# --- Sections 3-7: Train All Variants ------------------------------------------

def train_all_variants(dataset, variants, device):
    results = {}
    for variant in variants:
        print(f"\n{'='*72}")
        print(f"  VARIANT: {variant.upper()}")
        print(f"{'='*72}")

        train_loader, val_loader, info = create_dataloaders(
            dataset, image_size=IMAGE_SIZE,
            batch_size=VARIANT_BATCH_SIZES[variant],
            num_workers=8, aug_intensity="medium",
        )

        factory = {"nano": umobilevit_nano, "base": umobilevit_base,
                    "pro": umobilevit_pro, "promax": umobilevit_promax}
        model = factory[variant](out_channels=info.num_classes, head="single")
        n_params = sum(p.numel() for p in model.parameters())
        model = model.to(device)

        save_dir = f"./checkpoints/{dataset}_{variant}"
        os.makedirs(save_dir, exist_ok=True)

        use_amp = variant in ("nano", "base")
        grad_accum = 1 if variant in ("nano", "base") else 8

        trainer = SegmentationTrainer(
            model=model, device=device, dataset_info=info, save_dir=save_dir,
            config=TrainingConfig(
                lr=1e-3, weight_decay=1e-4, warmup_epochs=5,
                scheduler_type="poly", use_ema=True, ema_decay=0.999,
                use_amp=use_amp, grad_accum_steps=grad_accum,
                label_smoothing=0.1,
            ),
        )

        history = trainer.train(
            train_loader, val_loader,
            epochs=VARIANT_EPOCHS[variant],
            patience=20, min_epochs=30,
        )

        best_metric = max(history["val_metric"])
        metric_name = "Dice" if info.type == "binary" else "mIoU"
        print(f"  Best {metric_name}: {best_metric:.4f} ({n_params/1e6:.2f}M params)")

        results[variant] = {
            "history": history, "n_params": n_params,
            "best_metric": best_metric, "trainer": trainer,
        }

        del model, trainer, train_loader, val_loader
        if device.type == "cuda":
            torch.cuda.empty_cache()

    return results

ablation_results = train_all_variants(DATASET, VARIANTS, device)



  VARIANT: NANO
[Checkpointing] enabled on 11 layers


/home/sanng/miniconda3/envs/vision_env/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:216: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
Epoch 1/300 [Train]:   0%|          | 0/251 [00:00<?, ?batch/s]/home/sanng/miniconda3/envs/vision_env/lib/python3.11/site-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


KeyboardInterrupt: 

In [ ]:
# --- Sections 8-10: Ablation Analysis ------------------------------------------

def plot_ablation_analysis(results):
    variants_list = sorted(results.keys(), key=lambda v: results[v]["n_params"])

    # --- Parameters vs Accuracy scatter ---
    params_vals = [results[v]["n_params"] / 1e6 for v in variants_list]
    metrics_vals = [results[v]["best_metric"] for v in variants_list]

    fig, ax = plt.subplots(figsize=(8, 6))
    colors = [VARIANT_COLORS[v] for v in variants_list]
    ax.scatter(params_vals, metrics_vals, c=colors, s=150, edgecolors="white",
               linewidth=1, zorder=3)
    for v, x, y in zip(variants_list, params_vals, metrics_vals):
        ax.annotate(v.upper(), (x, y), textcoords="offset points",
                     xytext=(8, 5), fontsize=10, fontweight="bold")
    ax.set_xlabel("Parameters (M)")
    ax.set_ylabel("Best Validation mIoU")
    ax.set_title("Accuracy-Efficiency Trade-off (COCO Tea Leaf)")
    ax.grid(True, ls="--", alpha=0.5)
    os.makedirs("paper_figures", exist_ok=True)
    plt.savefig("paper_figures/03_params_vs_accuracy.pdf")
    plt.show()

    # --- Convergence overlay ---
    fig, ax = plt.subplots(figsize=(10, 6))
    for v in variants_list:
        history = results[v]["history"]
        epochs = range(1, len(history["val_metric"]) + 1)
        ax.plot(epochs, history["val_metric"], color=VARIANT_COLORS[v],
                lw=2, label=f"{v.upper()} ({results[v]['n_params']/1e6:.1f}M)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation mIoU")
    ax.set_title("Convergence Comparison Across Variants")
    ax.legend()
    ax.grid(True, ls="--", alpha=0.5)
    plt.savefig("paper_figures/03_convergence_overlay.pdf")
    plt.show()

    # --- Efficiency bar chart ---
    efficiency = [results[v]["best_metric"] / (results[v]["n_params"] / 1e6)
                  for v in variants_list]
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar([v.upper() for v in variants_list], efficiency,
                  color=[VARIANT_COLORS[v] for v in variants_list],
                  edgecolor="white", lw=0.5)
    ax.set_ylabel("mIoU / M parameters")
    ax.set_title("Segmentation Efficiency")
    for bar, val in zip(bars, efficiency):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)
    ax.grid(True, axis="y", ls="--", alpha=0.5)
    plt.savefig("paper_figures/03_efficiency.pdf")
    plt.show()

    # --- Summary table ---
    print(f"\n{'='*72}")
    print(f"  ABLATION SUMMARY | {DATASET}")
    print(f"{'='*72}")
    print(f"  {'Variant':<10s} {'Params (M)':>10s}  {'mIoU':>8s}  {'Efficiency':>10s}")
    print(f"  {'-'*48}")
    for v in variants_list:
        r = results[v]
        eff = r["best_metric"] / (r["n_params"] / 1e6)
        print(f"  {v.upper():<10s} {r['n_params']/1e6:>10.2f}  {r['best_metric']:>8.4f}  {eff:>10.3f}")
    print(f"{'='*72}")

plot_ablation_analysis(ablation_results)
